# Phase 2 — Corpus et tokenizer

Ce notebook prépare tout ce dont l'entraînement aura besoin :
1. télécharger un corpus français (Wikipédia) ;
2. le nettoyer ;
3. entraîner un vrai tokenizer BPE (le même principe que `bpe_v2.py`, mais en Rust) ;
4. transformer tout le corpus en nombres, enregistrés sur ton Google Drive.

**Avant de commencer :** cette phase n'a pas besoin de GPU. Exécution → Modifier le type d'exécution → **CPU**. Garde ton quota de GPU pour l'entraînement.

## 1. Connexion à Google Drive
Tout ce qu'on produit est enregistré sur ton Drive : si Colab se déconnecte, rien n'est perdu.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DOSSIER = "/content/drive/MyDrive/petit-llm"   # tout le projet vivra ici
os.makedirs(DOSSIER, exist_ok=True)

!pip -q install datasets tokenizers

Mounted at /content/drive


## 2. Le corpus : Wikipédia en français

On lit Wikipédia **en streaming** : les articles arrivent un par un, sans tout télécharger (le dump français fait plusieurs Go).

`shuffle` mélange les articles, sinon on n'aurait que ceux du début de l'ordre alphabétique des identifiants.

Nettoyage :
- on écarte les articles trop courts (ébauches, pages presque vides) ;
- **on garde les majuscules** cette fois. Un vrai modèle doit savoir écrire « Paris » et pas « paris ».

👉 **Pour un premier essai**, mets `NB_CARACTERES = 50_000_000` : ça prend quelques minutes et tu vérifies que tout marche. Ensuite, relance avec `2_000_000_000` (environ 500 millions de tokens).

In [10]:
import json, time
from datasets import load_dataset

NB_CARACTERES = 2_000_000_000
TAILLE_MIN = 500                # on écarte les articles de moins de 500 caractères

ds = load_dataset("wikimedia/wikipedia", "20231101.fr", split="train", streaming=True)
ds = ds.shuffle(seed=42, buffer_size=10_000)

CORPUS = "/content/corpus.jsonl"   # en local : bien plus rapide que le Drive
total, gardes, ecartes = 0, 0, 0
t0 = time.time()
with open(CORPUS, "w", encoding="utf-8") as f:
    for article in ds:
        texte = article["text"]
        for marque in ["\nNotes et références", "\nRéférences", "\nLiens externes",
                       "\nVoir aussi", "\nBibliographie"]:
            pos = texte.find(marque)
            if pos != -1:
                texte = texte[:pos]      # on garde seulement ce qui précède
        texte = texte.strip()
        if len(texte) < TAILLE_MIN:
            ecartes += 1
            continue
        f.write(json.dumps(texte, ensure_ascii=False) + "\n")
        gardes += 1
        total += len(texte)
        if gardes % 20_000 == 0:
            print(f"{gardes:,} articles, {total / 1e6:,.0f} M caractères, {time.time() - t0:.0f} s")
        if total >= NB_CARACTERES:
            break

print(f"\nArticles gardés : {gardes:,} | écartés (trop courts) : {ecartes:,}")
print(f"Corpus : {total:,} caractères")

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

20,000 articles, 55 M caractères, 15 s
40,000 articles, 113 M caractères, 26 s
60,000 articles, 169 M caractères, 36 s
80,000 articles, 227 M caractères, 47 s
100,000 articles, 289 M caractères, 62 s
120,000 articles, 345 M caractères, 76 s
140,000 articles, 401 M caractères, 87 s
160,000 articles, 459 M caractères, 98 s
180,000 articles, 514 M caractères, 110 s
200,000 articles, 569 M caractères, 124 s
220,000 articles, 623 M caractères, 136 s
240,000 articles, 676 M caractères, 151 s
260,000 articles, 731 M caractères, 162 s
280,000 articles, 761 M caractères, 171 s
300,000 articles, 805 M caractères, 186 s
320,000 articles, 870 M caractères, 199 s
340,000 articles, 931 M caractères, 209 s
360,000 articles, 986 M caractères, 222 s
380,000 articles, 1,046 M caractères, 234 s
400,000 articles, 1,105 M caractères, 248 s
420,000 articles, 1,161 M caractères, 261 s
440,000 articles, 1,220 M caractères, 271 s
460,000 articles, 1,278 M caractères, 283 s
480,000 articles, 1,334 M caractères,

## 3. Entraîner le tokenizer

Même algorithme que ton `bpe_v2.py`, avec trois différences de « vrai » tokenizer :
- **Byte-level** : il part des *octets* et non des caractères. Il ne rencontrera jamais de caractère inconnu (emoji, chinois…) : au pire, il redescend aux octets.
- **16 384 tokens** : un bon compromis pour un petit modèle (on a vu pourquoi : plus de tokens = séquences plus courtes, mais tokens plus rares et une table de vecteurs plus grosse à apprendre).
- un **token spécial `<|endoftext|>`** qui marque la fin de chaque article, pour que le modèle sache qu'un texte est terminé.

On l'entraîne sur les 300 premiers millions de caractères : c'est largement assez pour trouver les bonnes fusions.

In [11]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

TAILLE_VOCAB = 16_384
FIN = "<|endoftext|>"

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

entraineur = trainers.BpeTrainer(
    vocab_size=TAILLE_VOCAB,
    special_tokens=[FIN],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    min_frequency=2,
)

def lire_corpus(limite=None):
    lus = 0
    with open(CORPUS, encoding="utf-8") as f:
        for ligne in f:
            texte = json.loads(ligne)
            yield texte
            lus += len(texte)
            if limite and lus >= limite:
                return

t0 = time.time()
tokenizer.train_from_iterator(lire_corpus(300_000_000), trainer=entraineur)
tokenizer.save(f"{DOSSIER}/tokenizer.json")
print(f"Tokenizer entraîné en {time.time() - t0:.0f} s : {tokenizer.get_vocab_size():,} tokens")

Tokenizer entraîné en 210 s : 16,384 tokens


## 4. Tester le tokenizer
Compare avec ce que donnait ton `bpe_v2.py` sur la même phrase (13 tokens).

In [12]:
phrase = "Jean Valjean regardait l'évêque avec une étrange tranquillité."
enc = tokenizer.encode(phrase)
print("Découpage :", "|".join(tokenizer.decode([i]) for i in enc.ids))
print(f"{len(phrase)} caractères -> {len(enc.ids)} tokens")
print("Décodage correct :", tokenizer.decode(enc.ids) == phrase)

# Compression mesurée sur 1 000 articles
n_car, n_tok = 0, 0
for i, texte in enumerate(lire_corpus()):
    n_car += len(texte)
    n_tok += len(tokenizer.encode(texte).ids)
    if i == 1000:
        break
print(f"\nCompression : {n_car / n_tok:.2f} caractères par token")

# Quelques tokens longs appris sur Wikipédia
vocab = tokenizer.get_vocab()
longs = sorted(vocab, key=lambda t: len(tokenizer.decode([vocab[t]])), reverse=True)[:20]
print("Tokens les plus longs :", [tokenizer.decode([vocab[t]]) for t in longs])

Découpage : Jean| Val|je|an| regard|ait| l|'|évêque| avec| une| étr|ange| tran|qu|ill|ité|.
62 caractères -> 18 tokens
Décodage correct : True

Compression : 3.73 caractères par token
Tokens les plus longs : ['\n                       ', '\n                   ', ' traditionnellement', ' vraisemblablement', ' constitutionnelle', ' cinématographique', ' particulièrement', ' considérablement', ' professionnelles', ' circonscriptions', '\n                ', 'approvisionnement', ' caractéristiques', ' traditionnelles', ' pseudoscorpions', ' caractéristique', ' progressivement', ' enregistrements', ' révolutionnaire', ' professionnelle']


## 5. Tout le corpus en nombres

On encode chaque article, on ajoute `<|endoftext|>` à la fin, et on écrit les ids dans un fichier binaire (`uint16` : 2 octets par token, possible parce que 16 384 < 65 536).

- **`train.bin`** : ce sur quoi le modèle apprend ;
- **`val.bin`** : 1 article sur 200, mis de côté. Le modèle ne les verra **jamais** pendant l'entraînement : c'est sur eux qu'on vérifiera s'il a vraiment appris le français ou s'il récite (souviens-toi de ton N = 8).

In [13]:
import numpy as np, shutil

FIN_ID = tokenizer.token_to_id(FIN)
LOT = 2_000
n_train, n_val, n_doc = 0, 0, 0

t0 = time.time()
with open("/content/train.bin", "wb") as f_train, open("/content/val.bin", "wb") as f_val:
    lot = []
    def ecrire(lot):
        global n_train, n_val, n_doc
        for enc in tokenizer.encode_batch(lot):
            ids = np.array(enc.ids + [FIN_ID], dtype=np.uint16)
            if n_doc % 200 == 0:
                ids.tofile(f_val);   n_val += len(ids)
            else:
                ids.tofile(f_train); n_train += len(ids)
            n_doc += 1

    for texte in lire_corpus():
        lot.append(texte)
        if len(lot) == LOT:
            ecrire(lot); lot = []
            if n_doc % 100_000 == 0:
                print(f"{n_doc:,} articles, {n_train / 1e6:,.0f} M tokens, {time.time() - t0:.0f} s")
    if lot:
        ecrire(lot)

print(f"\ntrain : {n_train:,} tokens | val : {n_val:,} tokens | {time.time() - t0:.0f} s")

# Copie sur le Drive
for nom in ["train.bin", "val.bin"]:
    shutil.copy(f"/content/{nom}", f"{DOSSIER}/{nom}")
print("Copié dans", DOSSIER, ":", os.listdir(DOSSIER))

100,000 articles, 76 M tokens, 277 s
200,000 articles, 151 M tokens, 541 s
300,000 articles, 214 M tokens, 765 s
400,000 articles, 296 M tokens, 1056 s
500,000 articles, 372 M tokens, 1327 s
600,000 articles, 448 M tokens, 1592 s
700,000 articles, 526 M tokens, 1855 s

train : 536,800,272 tokens | val : 2,770,472 tokens | 1892 s
Copié dans /content/drive/MyDrive/petit-llm : ['tokenizer.json', 'train.bin', 'val.bin']


## 6. Vérification
On relit le fichier binaire et on décode un morceau au hasard : on doit retomber sur du texte de Wikipédia.

In [14]:
donnees = np.memmap(f"{DOSSIER}/train.bin", dtype=np.uint16, mode="r")
print(f"{len(donnees):,} tokens dans train.bin")
debut = np.random.randint(0, len(donnees) - 100)
print("\nExtrait décodé :\n", tokenizer.decode(donnees[debut:debut + 100].tolist()))

536,800,272 tokens dans train.bin

Extrait décodé :
 enfoncer dans les régions totalement nomades et sans habitations.
Le ksar de M'hamid est décrit par le célèbre explorateur René Caillié dans son ouvrage relatant son voyage à Tombouctou en 1830.

Administration et politique 
Dans le cadre de l'administration territoriale marocaine, la commune de M'Hamid El Ghizlane est une collectivité territoriale de la province de Zagora, dans la région de Dra
